In [0]:
# Read raw CSV file


bronze_df = spark.table("narayana.default.students")

bronze_df.show()





+----------+--------+-------+-----+----------+
|student_id|    name| course|score|attendance|
+----------+--------+-------+-----+----------+
|         1|    John|   Math|   80|        90|
|         2|   Alice|Science|   75|        85|
|         3|     Bob|   Math|   90|        95|
|         4|   David|English|   70|        80|
|         5|    Emma|Science|   88|        92|
|         6|   Frank|   Math|   65|        75|
|         7|   Grace|English|   95|        98|
|         8|   Henry|Science| NULL|        89|
|         9|Isabella|   Math|   78|        91|
|        10|    Jack|English|   82|        87|
|        11|    John|   Math|   80|        90|
+----------+--------+-------+-----+----------+



In [0]:
# Count the number of records in table

bronze_df.count()

11

In [0]:
# Save to Bronze Delta Table

bronze_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_students")

In [0]:
%sql
SELECT * FROM bronze_students;

student_id,name,course,score,attendance
1,John,Math,80,90
2,Alice,Science,75,85
3,Bob,Math,90,95
4,David,English,70,80
5,Emma,Science,88,92
6,Frank,Math,65,75
7,Grace,English,95,98
8,Henry,Science,null,89
9,Isabella,Math,78,91
10,Jack,English,82,87


In [0]:
# Read Bronze Table

bronze_df = spark.table("bronze_students")

In [0]:
# Remove Null Scores

silver_df = bronze_df.dropDuplicates()

In [0]:
# Remove Null Scores

silver_df = silver_df.filter(
    silver_df.score.isNotNull()
)

In [0]:
# Accept only scores between 0 and 100

silver_df = silver_df.filter(
    (silver_df.score >= 0) &
    (silver_df.score <= 100)
)

In [0]:
# Save to Silver Delta Table

silver_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_students")

In [0]:
%sql
SELECT * FROM silver_students;

student_id,name,course,score,attendance
10,Jack,English,82,87
4,David,English,70,80
2,Alice,Science,75,85
9,Isabella,Math,78,91
1,John,Math,80,90
6,Frank,Math,65,75
11,John,Math,80,90
5,Emma,Science,88,92
7,Grace,English,95,98
3,Bob,Math,90,95


In [0]:
# Aggregation Code

from pyspark.sql.functions import avg,count

gold_df = silver_df.groupBy("course") \
    .agg(
        avg("score").alias("avg_score"),
        avg("attendance").alias("avg_attendance"),
        count("*").alias("student_count")
    )

In [0]:
# Save to Gold Delta Table

gold_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_student_performance")

In [0]:
%sql
SELECT * FROM gold_student_performance;

course,avg_score,avg_attendance,student_count
English,82.33333333333333,88.33333333333333,3
Math,78.6,88.2,5
Science,81.5,88.5,2


In [0]:
%sql
SELECT
course,
avg_score
FROM gold_student_performance;

course,avg_score
English,82.33333333333333
Math,78.6
Science,81.5


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT
course,
student_count
FROM gold_student_performance;

course,student_count
English,3
Math,5
Science,2


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT
course,
avg_attendance
FROM gold_student_performance;

course,avg_attendance
English,88.33333333333333
Math,88.2
Science,88.5


Databricks visualization. Run in Databricks to view.